In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import zipfile
import os

zip_file_path = 'results.zip'

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall('/content/')


os.chdir('/content/')

In [ ]:
# --- imports & plotting defaults ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.gridspec import GridSpec

# style
matplotlib.rcParams.update({
    "axes.titlesize": 18,
    "axes.labelsize": 22,     # ← controls x/y label font size
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "font.size": 18,
    "legend.fontsize": 18,
    "lines.linewidth": 2,
    "font.weight": "normal",
    "lines.markersize": 10,
    "lines.markerfacecolor": "none",
    "text.latex.preamble": r"\usepackage{amsfonts}",
})
plt.rcParams["mathtext.fontset"] = "cm"
plt.rc("text", usetex=False)
plt.rc("font", family="serif")

# Binary LSAC: Sensitive attribute gender

In [ ]:
# --- helper functions ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

def load_metric_values(base_dir, dataset, ml_model, split_strategy, folder, metric, eps=None):
    """
    Load all numeric values for a given metric.
    If eps is None, load the non_private file.
    Returns a 1D numpy array of values.
    """
    if eps is None:
        f = f"{base_dir}/{dataset}/non_private/{ml_model}_results_non_private.csv"
    else:
        f = f"{base_dir}/{dataset}/{folder}/{split_strategy}/Appendix_{ml_model}_results_{folder}_eps_{eps}.csv"
    df = pd.read_csv(f, usecols=[metric])
    vals = pd.to_numeric(df[metric], errors="coerce").dropna().astype(float).values
    return vals


def mean_ci(values, z=1.96):
    """Return (mean, half_width) for a 95% CI using z*SE (ddof=1)."""
    values = np.asarray(values, dtype=float)
    n = len(values)
    if n == 0:
        return np.nan, np.nan
    m = np.mean(values)
    if n == 1:
        return m, 0.0
    s = np.std(values, ddof=1)
    return m, z * s / np.sqrt(n)


def aggregate_over_eps(base_dir, dataset, ml_model, split_strategy, folder, metric, eps_list):
    means, cis = [], []
    for eps in eps_list:
        vals = load_metric_values(base_dir, dataset, ml_model, split_strategy, folder, metric, eps)
        m, ci = mean_ci(vals)
        means.append(m); cis.append(ci)
    return np.array(means), np.array(cis)



# ---- config ----
base_dir       = "results"
datasets       = ["LSAC"]
ml_model       = "LGBM"
split_strategy = "uniform"
eps_list       = [0.05, 0.25, 0.5, 1, 2, 4, 8]

utility_metric    = "acc"
fairness_metrics  = ["SPD", "EOD"]

# Colors and markers
markers = {"OPT": "o", "GRR": "s", "SS": "^"}
colors  = {"OPT": "#E67E22",  # orange
           "GRR": "#2CA02C",  # green
           "SS":  "#1F77B4"}  # blue

# Legend display names
legend_names = {"OPT": "OPT", "GRR": "RR", "SS": "SS"}

fair_labels = {
    "SPD": r"$\Delta_{\mathrm{SP}}$",
    "EOD": r"$\Delta_{\mathrm{EO}}$",
}

# Manually set accuracy y-axis range
ACC_YLIM = (0.94, 0.9475)

for dataset in datasets:
    # --- baselines ---
    non_dp_acc_vals = load_metric_values(
        base_dir, dataset, ml_model, split_strategy, folder="non_private",
        metric=utility_metric, eps=None
    )
    non_dp_acc_mean, _ = mean_ci(non_dp_acc_vals)

    non_dp_fair_means = {}
    for fm in fairness_metrics:
        vals = load_metric_values(
            base_dir, dataset, ml_model, split_strategy, folder="non_private",
            metric=fm, eps=None
        )
        non_dp_fair_means[fm], _ = mean_ci(vals)

    # --- aggregate curves for mechanisms ---
    mechs = ["OPT", "GRR", "SS"]
    agg = {}
    for mech in mechs:
        agg[(mech, utility_metric)] = aggregate_over_eps(
            base_dir, dataset, ml_model, split_strategy, mech, utility_metric, eps_list
        )
        for fm in fairness_metrics:
            agg[(mech, fm)] = aggregate_over_eps(
                base_dir, dataset, ml_model, split_strategy, mech, fm, eps_list
            )

    # --- layout: left wide for accuracy, right split into 2 rows for fairness ---
    fig = plt.figure(figsize=(15, 7))
    gs = GridSpec(nrows=2, ncols=2, width_ratios=[1.4, 1.0], height_ratios=[1, 1], figure=fig)

    ax_u   = fig.add_subplot(gs[:, 0])  # left big panel spans both rows
    ax_spd = fig.add_subplot(gs[0, 1])  # top-right
    ax_eod = fig.add_subplot(gs[1, 1])  # bottom-right

    x = np.arange(len(eps_list))
    xlabels = [str(e) for e in eps_list]

    # ===== Left: Accuracy =====
    ax_u.grid(True, linestyle="dashdot", linewidth=0.5)
    ax_u.axhline(non_dp_acc_mean, color="black", linestyle="dashed",
                 linewidth=1.4, label="NonDP")

    for mech in mechs:
        means, cis = agg[(mech, utility_metric)]
        c = colors[mech]
        ax_u.plot(x, means, marker=markers[mech], linestyle="solid",
                  label=legend_names[mech], color=c)
        ax_u.fill_between(x, means - cis, means + cis, alpha=0.20, linewidth=0, color=c)

    ax_u.set_xticks(x)
    ax_u.set_xticklabels(xlabels)
    ax_u.set_xlabel(r"$\varepsilon$")
    ax_u.set_ylabel("Accuracy")
    ax_u.set_ylim(*ACC_YLIM)   # apply manual interval

    # ===== Right-top: SPD =====
    fm = "SPD"
    ax_spd.grid(True, linestyle="dashdot", linewidth=0.5)
    ax_spd.axhline(non_dp_fair_means[fm], color="black", linestyle="dashed", linewidth=1.2)
    for mech in mechs:
        means, cis = agg[(mech, fm)]
        c = colors[mech]
        ax_spd.plot(x, means, marker=markers[mech], linestyle="solid", color=c)
        ax_spd.fill_between(x, means - cis, means + cis, alpha=0.15, linewidth=0, color=c)
    ax_spd.set_xticks(x)
    ax_spd.set_xticklabels([])
    ax_spd.set_ylabel(fair_labels[fm])

    # ===== Right-bottom: EOD =====
    fm = "EOD"
    ax_eod.grid(True, linestyle="dashdot", linewidth=0.5)
    ax_eod.axhline(non_dp_fair_means[fm], color="black", linestyle="dashed", linewidth=1.2)
    for mech in mechs:
        means, cis = agg[(mech, fm)]
        c = colors[mech]
        ax_eod.plot(x, means, marker=markers[mech], linestyle="solid", color=c)
        ax_eod.fill_between(x, means - cis, means + cis, alpha=0.15, linewidth=0, color=c)
    ax_eod.set_xticks(x)
    ax_eod.set_xticklabels(xlabels)
    ax_eod.set_xlabel(r"$\varepsilon$")
    ax_eod.set_ylabel(fair_labels[fm])

    # ===== legend  =====
    handles, labels = ax_u.get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.02),
               ncol=4, frameon=False)

    plt.tight_layout(rect=[0, 0, 1, 0.98])

    # Save
    png_out = f"{base_dir}/fig_acc_spd_eod_{dataset}_final.png"
    pdf_out = f"{base_dir}/fig_acc_spd_eod_{dataset}_final.pdf"
    fig.savefig(png_out, dpi=600, bbox_inches="tight", pad_inches=0.1)
    fig.savefig(pdf_out, dpi=600, bbox_inches="tight", pad_inches=0.1)
    plt.show()

    print(f"Saved: {png_out}\n       {pdf_out}")


# Binary Adult: Sensitive attribute gender

In [ ]:
# --- helper functions ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

def load_metric_values(base_dir, dataset, ml_model, split_strategy, folder, metric, eps=None):
    """
    Load numeric values for a given metric.
    If eps is None, load the non_private file.
    Returns a 1D numpy array of values.
    """
    if eps is None:
        f = f"{base_dir}/{dataset}/non_private/{ml_model}_results_non_private.csv"
    else:
        f = f"{base_dir}/{dataset}/{folder}/{split_strategy}/Appendix_{ml_model}_results_{folder}_eps_{eps}.csv"
    df = pd.read_csv(f, usecols=[metric])
    vals = pd.to_numeric(df[metric], errors="coerce").dropna().astype(float).values
    return vals

def mean_ci(values, z=1.96):
    """Return (mean, half_width) for a 95% CI using z*SE (ddof=1)."""
    values = np.asarray(values, dtype=float)
    n = len(values)
    if n == 0:
        return np.nan, np.nan
    m = np.mean(values)
    if n == 1:
        return m, 0.0
    s = np.std(values, ddof=1)
    return m, z * s / np.sqrt(n)

def aggregate_over_eps(base_dir, dataset, ml_model, split_strategy, folder, metric, eps_list):
    means, cis = [], []
    for eps in eps_list:
        vals = load_metric_values(base_dir, dataset, ml_model, split_strategy, folder, metric, eps)
        m, ci = mean_ci(vals)
        means.append(m); cis.append(ci)
    return np.array(means), np.array(cis)


# ---- config ----
base_dir       = "results"
datasets       = ["adult"]
ml_model       = "LGBM"
split_strategy = "uniform"
eps_list       = [0.05, 0.25, 0.5, 1, 2, 8, 10]

utility_metric    = "acc"
fairness_metrics  = ["SPD", "EOD"]

display_names = {"OPT": "OPT", "GRR": "RR", "SS": "SS"}

# Markers/colors
markers = {"OPT": "o", "GRR": "s", "SS": "^"}
colors  = {"OPT": "#E67E22",  # orange
           "GRR": "#2CA02C",  # green
           "SS":  "#1F77B4"}  # blue

fair_labels = {
    "SPD": r"$\Delta_{\mathrm{SP}}$",
    "EOD": r"$\Delta_{\mathrm{EO}}$",
}

# Optional manual y-lims (keep or tweak; set to None to autoscale)
ACC_YLIM   = (0.81, 0.82)              # e.g., (0.84, 0.90)
FAIR_YLIMS = {
    "SPD": (0.32, 0.38),
    "EOD": (0.16, 0.23),
}

for dataset in datasets:
    # --- baselines (NonDP) ---
    non_dp_acc_vals = load_metric_values(
        base_dir, dataset, ml_model, split_strategy, folder="non_private",
        metric=utility_metric, eps=None
    )
    non_dp_acc_mean, _ = mean_ci(non_dp_acc_vals)

    non_dp_fair_means = {}
    for fm in fairness_metrics:
        vals = load_metric_values(
            base_dir, dataset, ml_model, split_strategy, folder="non_private",
            metric=fm, eps=None
        )
        non_dp_fair_means[fm], _ = mean_ci(vals)

    # --- aggregate curves for mechanisms ---
    mechs = ["OPT", "GRR", "SS"]
    agg = {}
    for mech in mechs:
        agg[(mech, utility_metric)] = aggregate_over_eps(
            base_dir, dataset, ml_model, split_strategy, mech, utility_metric, eps_list
        )
        for fm in fairness_metrics:
            agg[(mech, fm)] = aggregate_over_eps(
                base_dir, dataset, ml_model, split_strategy, mech, fm, eps_list
            )

    # --- layout: left wide for accuracy, right split into 2 rows for fairness ---
    fig = plt.figure(figsize=(15, 7))
    gs = GridSpec(nrows=2, ncols=2, width_ratios=[1.4, 1.0], height_ratios=[1, 1], figure=fig)

    ax_u   = fig.add_subplot(gs[:, 0])  # left big panel spans both rows
    ax_spd = fig.add_subplot(gs[0, 1])  # top-right
    ax_eod = fig.add_subplot(gs[1, 1])  # bottom-right

    x = np.arange(len(eps_list))
    xlabels = [str(e) for e in eps_list]

    # ===== Left: Accuracy =====
    ax_u.grid(True, linestyle="dashdot", linewidth=0.5)
    ax_u.axhline(non_dp_acc_mean, color="black", linestyle="dashed",
                 linewidth=1.4, label="NonDP")

    for mech in mechs:
        means, cis = agg[(mech, utility_metric)]
        c = colors[mech]
        ax_u.plot(x, means, marker=markers[mech], linestyle="solid",
                  label=display_names[mech], color=c)
        ax_u.fill_between(x, means - cis, means + cis, alpha=0.20, linewidth=0, color=c)

    ax_u.set_xticks(x)
    ax_u.set_xticklabels(xlabels)
    ax_u.set_xlabel(r"$\varepsilon$")
    ax_u.set_ylabel("Accuracy")
    if ACC_YLIM is not None:
        ax_u.set_ylim(*ACC_YLIM)

    # ===== Right-top: SPD =====
    fm = "SPD"
    ax_spd.grid(True, linestyle="dashdot", linewidth=0.5)
    ax_spd.axhline(non_dp_fair_means[fm], color="black", linestyle="dashed", linewidth=1.2)
    for mech in mechs:
        means, cis = agg[(mech, fm)]
        c = colors[mech]
        ax_spd.plot(x, means, marker=markers[mech], linestyle="solid", color=c)
        ax_spd.fill_between(x, means - cis, means + cis, alpha=0.15, linewidth=0, color=c)
    ax_spd.set_xticks(x)
    ax_spd.set_xticklabels([])
    ax_spd.set_ylabel(fair_labels[fm])
    if FAIR_YLIMS.get(fm) is not None:
        ax_spd.set_ylim(*FAIR_YLIMS[fm])

    # ===== Right-bottom: EOD =====
    fm = "EOD"
    ax_eod.grid(True, linestyle="dashdot", linewidth=0.5)
    ax_eod.axhline(non_dp_fair_means[fm], color="black", linestyle="dashed", linewidth=1.2)
    for mech in mechs:
        means, cis = agg[(mech, fm)]
        c = colors[mech]
        ax_eod.plot(x, means, marker=markers[mech], linestyle="solid", color=c)
        ax_eod.fill_between(x, means - cis, means + cis, alpha=0.15, linewidth=0, color=c)
    ax_eod.set_xticks(x)
    ax_eod.set_xticklabels(xlabels)
    ax_eod.set_xlabel(r"$\varepsilon$")
    ax_eod.set_ylabel(fair_labels[fm])
    if FAIR_YLIMS.get(fm) is not None:
        ax_eod.set_ylim(*FAIR_YLIMS[fm])

    # ===== legend  =====
    handles, labels = ax_u.get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.02),
               ncol=4, frameon=False)

    plt.tight_layout(rect=[0, 0, 1, 0.98])

    # Save
    png_out = f"{base_dir}/fig_acc_spd_eod_{dataset}_final.png"
    pdf_out = f"{base_dir}/fig_acc_spd_eod_{dataset}_final.pdf"
    fig.savefig(png_out, dpi=600, bbox_inches="tight", pad_inches=0.1)
    fig.savefig(pdf_out, dpi=600, bbox_inches="tight", pad_inches=0.1)
    plt.show()

    print(f"Saved: {png_out}\n       {pdf_out}")


# Non Binary LSAC: Sensitive attribute family income

In [ ]:
# --- helper functions ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

def load_metric_values(base_dir, dataset, ml_model, split_strategy, folder, metric, eps=None):
    """
    Load numeric values for a given metric.
    If eps is None, load the non_private file.
    Returns a 1D numpy array of values.
    """
    if eps is None:
        f = f"{base_dir}/{dataset}/non_private/{ml_model}_results_non_private.csv"
    else:
        f = f"{base_dir}/{dataset}/{folder}/{split_strategy}/Appendix_{ml_model}_results_{folder}_eps_{eps}.csv"
    df = pd.read_csv(f, usecols=[metric])
    vals = pd.to_numeric(df[metric], errors="coerce").dropna().astype(float).values
    return vals

def mean_ci(values, z=1.96):
    """Return (mean, half_width) for a 95% CI using z*SE (ddof=1)."""
    values = np.asarray(values, dtype=float)
    n = len(values)
    if n == 0:
        return np.nan, np.nan
    m = np.mean(values)
    if n == 1:
        return m, 0.0
    s = np.std(values, ddof=1)
    return m, z * s / np.sqrt(n)

def aggregate_over_eps(base_dir, dataset, ml_model, split_strategy, folder, metric, eps_list):
    means, cis = [], []
    for eps in eps_list:
        vals = load_metric_values(base_dir, dataset, ml_model, split_strategy, folder, metric, eps)
        m, ci = mean_ci(vals)
        means.append(m); cis.append(ci)
    return np.array(means), np.array(cis)

# --- single figure for LSAC: left=Utility (acc or f1), right=(SPD,EOD) stacked ---
base_dir       = "results"
datasets       = ["LSAC"]
ml_model       = "LGBM"
split_strategy = "uniform"
eps_list       = [0.25, 0.5, 1, 2, 4, 8, 10]


utility_metric    = "acc"
fairness_metrics  = ["SPD", "EOD"]

# Visuals consistent with your other figure
markers = {"OPT": "o", "GRR": "s", "SS": "^"}
colors  = {"OPT": "#E67E22",  # orange
           "GRR": "#2CA02C",  # green
           "SS":  "#1F77B4"}  # blue

fair_labels = {
    "SPD": r"$\Delta_{\mathrm{SP}}$",
    "EOD": r"$\Delta_{\mathrm{EO}}$",
}

# Optional manual y-lims
# Set to None to autoscale
UTIL_YLIM  = (0.941,.946)
FAIR_YLIMS = {
    "SPD": (0.03, 0.11),
    "EOD": (0.02, 0.10),
}

for dataset in datasets:
    # --- NonDP baselines from file means ---
    non_dp_util_vals = load_metric_values(
        base_dir, dataset, ml_model, split_strategy, folder="non_private",
        metric=utility_metric, eps=None
    )
    non_dp_util_mean, _ = mean_ci(non_dp_util_vals)

    non_dp_fair_means = {}
    for fm in fairness_metrics:
        vals = load_metric_values(
            base_dir, dataset, ml_model, split_strategy, folder="non_private",
            metric=fm, eps=None
        )
        non_dp_fair_means[fm], _ = mean_ci(vals)

    # --- aggregate curves for mechanisms ---
    mechs = ["OPT", "GRR", "SS"]
    agg = {}
    for mech in mechs:
        agg[(mech, utility_metric)] = aggregate_over_eps(
            base_dir, dataset, ml_model, split_strategy, mech, utility_metric, eps_list
        )
        for fm in fairness_metrics:
            agg[(mech, fm)] = aggregate_over_eps(
                base_dir, dataset, ml_model, split_strategy, mech, fm, eps_list
            )

    # --- layout: left wide for utility, right split into 2 rows for fairness ---
    fig = plt.figure(figsize=(15, 7))
    gs = GridSpec(nrows=2, ncols=2, width_ratios=[1.4, 1.0], height_ratios=[1, 1], figure=fig)

    ax_u   = fig.add_subplot(gs[:, 0])  # left big panel spans both rows
    ax_spd = fig.add_subplot(gs[0, 1])  # top-right
    ax_eod = fig.add_subplot(gs[1, 1])  # bottom-right

    x = np.arange(len(eps_list))
    xlabels = [str(e) for e in eps_list]

    # ===== Left: Utility (acc or f1) =====
    ax_u.grid(True, linestyle="dashdot", linewidth=0.5)
    ax_u.axhline(non_dp_util_mean, color="black", linestyle="dashed",
                 linewidth=1.4, label="NonDP")  # if you want the +0.0003 visual offset, add +0.0003 here

    for mech in mechs:
        means, cis = agg[(mech, utility_metric)]
        c = colors[mech]
        ax_u.plot(x, means, marker=markers[mech], linestyle="solid",
                  label=mech, color=c)
        ax_u.fill_between(x, means - cis, means + cis, alpha=0.20, linewidth=0, color=c)

    ax_u.set_xticks(x)
    ax_u.set_xticklabels(xlabels)
    ax_u.set_xlabel(r"$\varepsilon$")
    ax_u.set_ylabel("Accuracy" if utility_metric == "acc" else "F1")
    if UTIL_YLIM is not None:
        ax_u.set_ylim(*UTIL_YLIM)

    # ===== Right-top: SPD =====
    fm = "SPD"
    ax_spd.grid(True, linestyle="dashdot", linewidth=0.5)
    ax_spd.axhline(non_dp_fair_means[fm], color="black", linestyle="dashed", linewidth=1.2)

    for mech in mechs:
        means, cis = agg[(mech, fm)]
        c = colors[mech]
        ax_spd.plot(x, means, marker=markers[mech], linestyle="solid", color=c)
        ax_spd.fill_between(x, means - cis, means + cis, alpha=0.15, linewidth=0, color=c)
    ax_spd.set_xticks(x)
    ax_spd.set_xticklabels([])
    ax_spd.set_ylabel(fair_labels[fm])
    if FAIR_YLIMS.get(fm) is not None:
        ax_spd.set_ylim(*FAIR_YLIMS[fm])

    # ===== Right-bottom: EOD =====
    fm = "EOD"
    ax_eod.grid(True, linestyle="dashdot", linewidth=0.5)
    ax_eod.axhline(non_dp_fair_means[fm], color="black", linestyle="dashed", linewidth=1.2)

    for mech in mechs:
        means, cis = agg[(mech, fm)]
        c = colors[mech]
        ax_eod.plot(x, means, marker=markers[mech], linestyle="solid", color=c)
        ax_eod.fill_between(x, means - cis, means + cis, alpha=0.15, linewidth=0, color=c)
    ax_eod.set_xticks(x)
    ax_eod.set_xticklabels(xlabels)
    ax_eod.set_xlabel(r"$\varepsilon$")
    ax_eod.set_ylabel(fair_labels[fm])
    if FAIR_YLIMS.get(fm) is not None:
        ax_eod.set_ylim(*FAIR_YLIMS[fm])

    # =====  legend =====
    handles, labels = ax_u.get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.02),
               ncol=4, frameon=False)

    plt.tight_layout(rect=[0, 0, 1, 0.98])

    # Save
    png_out = f"{base_dir}/fig_util_spd_eod_{dataset}_final.png"
    pdf_out = f"{base_dir}/fig_util_spd_eod_{dataset}_final.pdf"
    fig.savefig(png_out, dpi=600, bbox_inches="tight", pad_inches=0.1)
    fig.savefig(pdf_out, dpi=600, bbox_inches="tight", pad_inches=0.1)
    plt.show()

    print(f"Saved: {png_out}\n       {pdf_out}")


# Non Binary Adult: Sensitive attribute race

In [ ]:
# --- helper functions ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

def load_metric_values(base_dir, dataset, ml_model, split_strategy, folder, metric, eps=None):
    """
    Load numeric values for a given metric.
    If eps is None, load the non_private file.
    Returns a 1D numpy array of values.
    """
    if eps is None:
        f = f"{base_dir}/{dataset}/non_private/{ml_model}_results_non_private.csv"
    else:
        f = f"{base_dir}/{dataset}/{folder}/{split_strategy}/Appendix_{ml_model}_results_{folder}_eps_{eps}.csv"
    df = pd.read_csv(f, usecols=[metric])
    vals = pd.to_numeric(df[metric], errors="coerce").dropna().astype(float).values
    return vals

def mean_ci(values, z=1.96):
    """Return (mean, half_width) for a 95% CI using z*SE (ddof=1)."""
    values = np.asarray(values, dtype=float)
    n = len(values)
    if n == 0:
        return np.nan, np.nan
    m = np.mean(values)
    if n == 1:
        return m, 0.0
    s = np.std(values, ddof=1)
    return m, z * s / np.sqrt(n)

def aggregate_over_eps(base_dir, dataset, ml_model, split_strategy, folder, metric, eps_list):
    means, cis = [], []
    for eps in eps_list:
        vals = load_metric_values(base_dir, dataset, ml_model, split_strategy, folder, metric, eps)
        m, ci = mean_ci(vals)
        means.append(m); cis.append(ci)
    return np.array(means), np.array(cis)

base_dir       = "results"
datasets       = ["adult"]
ml_model       = "LGBM"
split_strategy = "uniform"
eps_list       = [0.05, 0.25, 0.5, 1, 2, 8, 10]

# choose "acc" or "f1"
utility_metric    = "acc"
fairness_metrics  = ["SPD", "EOD"]

# visuals
markers = {"OPT": "o", "GRR": "s", "SS": "^"}
colors  = {"OPT": "#E67E22",  # orange
           "GRR": "#2CA02C",  # green
           "SS":  "#1F77B4"}  # blue

fair_labels = {
    "SPD": r"$\Delta_{\mathrm{SP}}$",
    "EOD": r"$\Delta_{\mathrm{EO}}$",
}

# optional manual y-lims (set to None to autoscale)
UTIL_YLIM  = (0.81,0.82)
FAIR_YLIMS = {
    "SPD": (0.2,0.32),
    "EOD": (0.15,0.35),
}

for dataset in datasets:
    # --- NonDP baselines from file means ---
    non_dp_util_vals = load_metric_values(
        base_dir, dataset, ml_model, split_strategy, folder="non_private",
        metric=utility_metric, eps=None
    )
    non_dp_util_mean, _ = mean_ci(non_dp_util_vals)

    non_dp_fair_means = {}
    for fm in fairness_metrics:
        vals = load_metric_values(
            base_dir, dataset, ml_model, split_strategy, folder="non_private",
            metric=fm, eps=None
        )
        non_dp_fair_means[fm], _ = mean_ci(vals)

    # --- aggregate curves for mechanisms ---
    mechs = ["OPT", "GRR", "SS"]
    agg = {}
    for mech in mechs:
        agg[(mech, utility_metric)] = aggregate_over_eps(
            base_dir, dataset, ml_model, split_strategy, mech, utility_metric, eps_list
        )
        for fm in fairness_metrics:
            agg[(mech, fm)] = aggregate_over_eps(
                base_dir, dataset, ml_model, split_strategy, mech, fm, eps_list
            )

    # --- layout: left wide for utility, right split into 2 rows for fairness ---
    fig = plt.figure(figsize=(15, 7))
    gs = GridSpec(nrows=2, ncols=2, width_ratios=[1.4, 1.0], height_ratios=[1, 1], figure=fig)

    ax_u   = fig.add_subplot(gs[:, 0])  # left big panel spans both rows
    ax_spd = fig.add_subplot(gs[0, 1])  # top-right
    ax_eod = fig.add_subplot(gs[1, 1])  # bottom-right

    x = np.arange(len(eps_list))
    xlabels = [str(e) for e in eps_list]

    # ===== Left: Utility (acc or f1) =====
    ax_u.grid(True, linestyle="dashdot", linewidth=0.5)
    ax_u.axhline(non_dp_util_mean, color="black", linestyle="dashed",
                 linewidth=1.4, label="NonDP")

    for mech in mechs:
        means, cis = agg[(mech, utility_metric)]
        c = colors[mech]
        ax_u.plot(x, means, marker=markers[mech], linestyle="solid",
                  label=mech, color=c)
        ax_u.fill_between(x, means - cis, means + cis, alpha=0.20, linewidth=0, color=c)

    ax_u.set_xticks(x)
    ax_u.set_xticklabels(xlabels)
    ax_u.set_xlabel(r"$\varepsilon$")
    ax_u.set_ylabel("Accuracy" if utility_metric == "acc" else "F1")
    if UTIL_YLIM is not None:
        ax_u.set_ylim(*UTIL_YLIM)

    # ===== Right-top: SPD =====
    fm = "SPD"
    ax_spd.grid(True, linestyle="dashdot", linewidth=0.5)
    ax_spd.axhline(non_dp_fair_means[fm], color="black", linestyle="dashed", linewidth=1.2)

    for mech in mechs:
        means, cis = agg[(mech, fm)]
        c = colors[mech]
        ax_spd.plot(x, means, marker=markers[mech], linestyle="solid", color=c)
        ax_spd.fill_between(x, means - cis, means + cis, alpha=0.15, linewidth=0, color=c)
    ax_spd.set_xticks(x)
    ax_spd.set_xticklabels([])
    ax_spd.set_ylabel(fair_labels[fm])
    if FAIR_YLIMS.get(fm) is not None:
        ax_spd.set_ylim(*FAIR_YLIMS[fm])

    # ===== Right-bottom: EOD =====
    fm = "EOD"
    ax_eod.grid(True, linestyle="dashdot", linewidth=0.5)
    ax_eod.axhline(non_dp_fair_means[fm], color="black", linestyle="dashed", linewidth=1.2)

    for mech in mechs:
        means, cis = agg[(mech, fm)]
        c = colors[mech]
        ax_eod.plot(x, means, marker=markers[mech], linestyle="solid", color=c)
        ax_eod.fill_between(x, means - cis, means + cis, alpha=0.15, linewidth=0, color=c)
    ax_eod.set_xticks(x)
    ax_eod.set_xticklabels(xlabels)
    ax_eod.set_xlabel(r"$\varepsilon$")
    ax_eod.set_ylabel(fair_labels[fm])
    if FAIR_YLIMS.get(fm) is not None:
        ax_eod.set_ylim(*FAIR_YLIMS[fm])

    # ===== legend =====
    handles, labels = ax_u.get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.02),
               ncol=4, frameon=False)

    plt.tight_layout(rect=[0, 0, 1, 0.98])

    # Save
    png_out = f"{base_dir}/fig_util_spd_eod_{dataset}_final.png"
    pdf_out = f"{base_dir}/fig_util_spd_eod_{dataset}_final.pdf"
    fig.savefig(png_out, dpi=600, bbox_inches="tight", pad_inches=0.1)
    fig.savefig(pdf_out, dpi=600, bbox_inches="tight", pad_inches=0.1)
    plt.show()

    print(f"Saved: {png_out}\n       {pdf_out}")

# Non Binary Adult: Sensitive attribute race-gender

In [ ]:
# --- helper functions ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

def load_metric_values(base_dir, dataset, ml_model, split_strategy, folder, metric, eps=None):
    """
    Load numeric values for a given metric.
    If eps is None, load the non_private file.
    Returns a 1D numpy array of values.
    """
    if eps is None:
        f = f"{base_dir}/{dataset}/non_private/{ml_model}_results_non_private.csv"
    else:
        f = f"{base_dir}/{dataset}/{folder}/{split_strategy}/Appendix_{ml_model}_results_{folder}_eps_{eps}.csv"
    df = pd.read_csv(f, usecols=[metric])
    vals = pd.to_numeric(df[metric], errors="coerce").dropna().astype(float).values
    return vals

def mean_ci(values, z=1.96):
    """Return (mean, half_width) for a 95% CI using z*SE (ddof=1)."""
    values = np.asarray(values, dtype=float)
    n = len(values)
    if n == 0:
        return np.nan, np.nan
    m = np.mean(values)
    if n == 1:
        return m, 0.0
    s = np.std(values, ddof=1)
    return m, z * s / np.sqrt(n)

def aggregate_over_eps(base_dir, dataset, ml_model, split_strategy, folder, metric, eps_list):
    means, cis = [], []
    for eps in eps_list:
        vals = load_metric_values(base_dir, dataset, ml_model, split_strategy, folder, metric, eps)
        m, ci = mean_ci(vals)
        means.append(m); cis.append(ci)
    return np.array(means), np.array(cis)


base_dir       = "results"
datasets       = ["adult"]
ml_model       = "LGBM"
split_strategy = "uniform"
eps_list       = [0.05, 0.25, 0.5, 1, 2, 8]

# choose "acc" or "f1"
utility_metric    = "acc"
fairness_metrics  = ["SPD", "EOD"]

markers = {"OPT": "o", "GRR": "s", "SS": "^"}
colors  = {"OPT": "#E67E22",  # orange
           "GRR": "#2CA02C",  # green
           "SS":  "#1F77B4"}  # blue

fair_labels = {
    "SPD": r"$\Delta_{\mathrm{SP}}$",
    "EOD": r"$\Delta_{\mathrm{EO}}$",
}

# optional manual y-lims (set to None to autoscale)
UTIL_YLIM  = (0.81,0.82)
FAIR_YLIMS = {
    "SPD": (0.4, 0.6),
    "EOD": (0.3, 0.6),
}

for dataset in datasets:
    # --- NonDP baselines from file means ---
    non_dp_util_vals = load_metric_values(
        base_dir, dataset, ml_model, split_strategy, folder="non_private",
        metric=utility_metric, eps=None
    )
    non_dp_util_mean, _ = mean_ci(non_dp_util_vals)

    non_dp_fair_means = {}
    for fm in fairness_metrics:
        vals = load_metric_values(
            base_dir, dataset, ml_model, split_strategy, folder="non_private",
            metric=fm, eps=None
        )
        non_dp_fair_means[fm], _ = mean_ci(vals)

    # --- aggregate curves for mechanisms ---
    mechs = ["OPT", "GRR", "SS"]
    agg = {}
    for mech in mechs:
        agg[(mech, utility_metric)] = aggregate_over_eps(
            base_dir, dataset, ml_model, split_strategy, mech, utility_metric, eps_list
        )
        for fm in fairness_metrics:
            agg[(mech, fm)] = aggregate_over_eps(
                base_dir, dataset, ml_model, split_strategy, mech, fm, eps_list
            )

    # --- layout: left wide for utility, right split into 2 rows for fairness ---
    fig = plt.figure(figsize=(15, 7))
    gs = GridSpec(nrows=2, ncols=2, width_ratios=[1.4, 1.0], height_ratios=[1, 1], figure=fig)

    ax_u   = fig.add_subplot(gs[:, 0])  # left big panel spans both rows
    ax_spd = fig.add_subplot(gs[0, 1])  # top-right
    ax_eod = fig.add_subplot(gs[1, 1])  # bottom-right

    x = np.arange(len(eps_list))
    xlabels = [str(e) for e in eps_list]

    # ===== Left: Utility (acc or f1) =====
    ax_u.grid(True, linestyle="dashdot", linewidth=0.5)
    ax_u.axhline(non_dp_util_mean, color="black", linestyle="dashed",
                 linewidth=1.4, label="NonDP")

    for mech in mechs:
        means, cis = agg[(mech, utility_metric)]
        c = colors[mech]
        ax_u.plot(x, means, marker=markers[mech], linestyle="solid",
                  label=mech, color=c)
        ax_u.fill_between(x, means - cis, means + cis, alpha=0.20, linewidth=0, color=c)

    ax_u.set_xticks(x)
    ax_u.set_xticklabels(xlabels)
    ax_u.set_xlabel(r"$\varepsilon$")
    ax_u.set_ylabel("Accuracy" if utility_metric == "acc" else "F1")
    if UTIL_YLIM is not None:
        ax_u.set_ylim(*UTIL_YLIM)

    # ===== Right-top: SPD =====
    fm = "SPD"
    ax_spd.grid(True, linestyle="dashdot", linewidth=0.5)
    ax_spd.axhline(0.54, color="black", linestyle="dashed", linewidth=1.2)
    for mech in mechs:
        means, cis = agg[(mech, fm)]
        c = colors[mech]
        ax_spd.plot(x, means, marker=markers[mech], linestyle="solid", color=c)
        ax_spd.fill_between(x, means - cis, means + cis, alpha=0.15, linewidth=0, color=c)
    ax_spd.set_xticks(x)
    ax_spd.set_xticklabels([])
    ax_spd.set_ylabel(fair_labels[fm])
    if FAIR_YLIMS.get(fm) is not None:
        ax_spd.set_ylim(*FAIR_YLIMS[fm])

    # ===== Right-bottom: EOD =====
    fm = "EOD"
    ax_eod.grid(True, linestyle="dashdot", linewidth=0.5)
    ax_eod.axhline(non_dp_fair_means[fm], color="black", linestyle="dashed", linewidth=1.2)
    for mech in mechs:
        means, cis = agg[(mech, fm)]
        c = colors[mech]
        ax_eod.plot(x, means, marker=markers[mech], linestyle="solid", color=c)
        ax_eod.fill_between(x, means - cis, means + cis, alpha=0.15, linewidth=0, color=c)
    ax_eod.set_xticks(x)
    ax_eod.set_xticklabels(xlabels)
    ax_eod.set_xlabel(r"$\varepsilon$")
    ax_eod.set_ylabel(fair_labels[fm])
    if FAIR_YLIMS.get(fm) is not None:
        ax_eod.set_ylim(*FAIR_YLIMS[fm])


    handles, labels = ax_u.get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.02),
               ncol=4, frameon=False)

    plt.tight_layout(rect=[0, 0, 1, 0.98])

    # Save
    png_out = f"{base_dir}/fig_util_spd_eod_{dataset}_final.png"
    pdf_out = f"{base_dir}/fig_util_spd_eod_{dataset}_final.pdf"
    fig.savefig(png_out, dpi=600, bbox_inches="tight", pad_inches=0.1)
    fig.savefig(pdf_out, dpi=600, bbox_inches="tight", pad_inches=0.1)
    plt.show()

    print(f"Saved: {png_out}\n       {pdf_out}")
